# 📊 Dataset Comparison: SUN Database vs LDPolypVideo
### Deep Learning-Based Polyp Segmentation in Colonoscopy

This notebook trains the **same U-Net baseline** on both datasets and compares:
- **Training Loss** per epoch
- **Validation Accuracy**
- **Dice Coefficient**
- **IoU (Intersection over Union)**

## ⚙️ 0. Setup & Imports

In [ ]:
import os
import cv2
import glob
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, Subset
import torch.nn as nn
import torch.optim as optim

# Device
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

# ---- Paths ----
SUN_PATH      = '/Users/alokkumarshukla/Desktop/Major Project 1/Sun/sundatabase_positive_part1'
LD_TRAIN_PATH = '/Users/alokkumarshukla/Desktop/Major Project 1/LD Polyp/TrainValid'
LD_TEST_PATH  = '/Users/alokkumarshukla/Desktop/Major Project 1/LD Polyp/Test'

IMAGE_SIZE = (128, 128)
BATCH_SIZE = 8
EPOCHS     = 5
SUBSET     = 1000   # samples per dataset for fair comparison

## 📁 1. Dataset Loaders

In [ ]:
# ──────────────────────────────────────────────────
#  SUN Database Loader
#  Annotation format: "filename ymin,xmin,ymax,xmax,label"
# ──────────────────────────────────────────────────
class SUNPolypDataset(Dataset):
    def __init__(self, root_dir, image_size=IMAGE_SIZE):
        self.image_size = image_size
        self.samples = []
        anno_dir = os.path.join(root_dir, 'annotation_txt')
        for txt_file in glob.glob(os.path.join(anno_dir, '*.txt')):
            case_name = os.path.basename(txt_file).replace('.txt', '')
            case_dir  = os.path.join(root_dir, case_name)
            if not os.path.exists(case_dir): continue
            with open(txt_file, 'r') as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    parts = line.split(' ')
                    if len(parts) >= 2:
                        coords = parts[1].split(',')
                        if len(coords) >= 4:
                            box = [int(c) for c in coords[:4]]
                            img_path = os.path.join(case_dir, parts[0])
                            if os.path.exists(img_path):
                                self.samples.append((img_path, [box]))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, boxes = self.samples[idx]
        img  = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)
        for box in boxes:
            y1, x1, y2, x2 = box
            mask[min(y1,y2):max(y1,y2), min(x1,x2):max(x1,x2)] = 1
        img  = cv2.resize(img,  self.image_size).astype(np.float32) / 255.0
        mask = cv2.resize(mask, self.image_size, interpolation=cv2.INTER_NEAREST)
        return (torch.tensor(np.transpose(img, (2,0,1)), dtype=torch.float32),
                torch.tensor(mask, dtype=torch.float32).unsqueeze(0))


# ──────────────────────────────────────────────────
#  LDPolypVideo Loader
#  Annotation format per file:
#    Line 0 : number of polyps (0 = no polyp)
#    Lines 1+: "ymin xmin ymax xmax" per polyp
# ──────────────────────────────────────────────────
class LDPolypDataset(Dataset):
    def __init__(self, root_dir, image_size=IMAGE_SIZE, skip_empty=True):
        self.image_size = image_size
        self.samples = []
        img_root  = os.path.join(root_dir, 'Images')
        anno_root = os.path.join(root_dir, 'Annotations')
        for video_id in sorted(os.listdir(img_root)):
            img_dir  = os.path.join(img_root,  video_id)
            anno_dir = os.path.join(anno_root, video_id)
            if not os.path.isdir(img_dir): continue
            for img_file in sorted(glob.glob(os.path.join(img_dir, '*.jpg'))):
                frame_id   = os.path.splitext(os.path.basename(img_file))[0]
                anno_file  = os.path.join(anno_dir, frame_id + '.txt')
                if not os.path.exists(anno_file): continue
                with open(anno_file, 'r') as f:
                    lines = [l.strip().replace('\r','') for l in f if l.strip()]
                n_polyps = int(lines[0]) if lines else 0
                if skip_empty and n_polyps == 0: continue
                boxes = []
                for l in lines[1:n_polyps+1]:
                    coords = l.split()
                    if len(coords) >= 4:
                        boxes.append([int(c) for c in coords[:4]])
                self.samples.append((img_file, boxes))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, boxes = self.samples[idx]
        img  = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)
        for box in boxes:
            y1, x1, y2, x2 = box
            mask[min(y1,y2):max(y1,y2), min(x1,x2):max(x1,x2)] = 1
        img  = cv2.resize(img,  self.image_size).astype(np.float32) / 255.0
        mask = cv2.resize(mask, self.image_size, interpolation=cv2.INTER_NEAREST)
        return (torch.tensor(np.transpose(img, (2,0,1)), dtype=torch.float32),
                torch.tensor(mask, dtype=torch.float32).unsqueeze(0))


# Load datasets
sun_dataset = SUNPolypDataset(SUN_PATH)
ld_dataset  = LDPolypDataset(LD_TRAIN_PATH)

print(f'SUN Dataset        : {len(sun_dataset):,} annotated frames')
print(f'LDPolyp (TrainVal) : {len(ld_dataset):,} annotated frames')

## 👁️ 2. Visualization — One Sample from Each Dataset

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Dataset Sample Comparison', fontsize=16, fontweight='bold')

for row, (ds, name) in enumerate([(sun_dataset, 'SUN Database'), (ld_dataset, 'LDPolypVideo')]):
    img, mask = ds[0]
    axes[row, 0].imshow(img.permute(1, 2, 0).numpy())
    axes[row, 0].set_title(f'{name} — Frame')
    axes[row, 0].axis('off')
    axes[row, 1].imshow(mask.squeeze().numpy(), cmap='hot')
    axes[row, 1].set_title(f'{name} — Pseudo-Mask (Bounding Box)')
    axes[row, 1].axis('off')

plt.tight_layout()
plt.show()

## 🧠 3. U-Net Model & Metrics (Shared for Both)

In [ ]:
def calculate_metrics(pred, target, threshold=0.5):
    pred   = (torch.sigmoid(pred) > threshold).float()
    target = target.float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum()
    dice  = (2. * intersection + 1e-6) / (union + 1e-6)
    iou   = (intersection + 1e-6) / (union - intersection + 1e-6)
    acc   = (pred == target).sum() / torch.numel(pred)
    return acc.item(), dice.item(), iou.item()


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.conv(x)

class BasicUNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1):
        super().__init__()
        self.inc      = DoubleConv(n_channels, 64)
        self.down1    = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2    = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.up1      = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv_up1 = DoubleConv(256, 128)
        self.up2      = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv_up2 = DoubleConv(128, 64)
        self.outc     = nn.Conv2d(64, n_classes, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x  = self.conv_up1(torch.cat([x2, self.up1(x3)], dim=1))
        x  = self.conv_up2(torch.cat([x1, self.up2(x)],  dim=1))
        return self.outc(x)

print('U-Net architecture defined.')

## 🏋️ 4. Training Function (Reusable)

In [ ]:
def train_and_evaluate(dataset, dataset_name, subset_size=SUBSET, epochs=EPOCHS):
    print(f'\n{"="*60}')
    print(f'  Training on: {dataset_name}  ({subset_size} samples)')
    print(f'{"="*60}')

    # ── Subset & split ──
    indices = list(range(min(subset_size, len(dataset))))
    sub = Subset(dataset, indices)
    train_n = int(0.8 * len(sub))
    val_n   = len(sub) - train_n
    train_ds, val_ds = torch.utils.data.random_split(sub, [train_n, val_n])
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # ── Fresh model for each dataset ──
    model     = BasicUNet().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    history = {'loss': [], 'acc': [], 'dice': [], 'iou': []}

    for epoch in range(epochs):
        # Training
        model.train()
        total_loss = 0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), masks)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Validation
        model.eval()
        v_acc = v_dice = v_iou = 0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                a, d, i = calculate_metrics(model(imgs), masks)
                v_acc  += a;  v_dice += d;  v_iou += i

        avg_loss = total_loss / len(train_loader)
        avg_acc  = v_acc  / len(val_loader)
        avg_dice = v_dice / len(val_loader)
        avg_iou  = v_iou  / len(val_loader)

        history['loss'].append(avg_loss)
        history['acc'].append(avg_acc)
        history['dice'].append(avg_dice)
        history['iou'].append(avg_iou)

        print(f'  Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Acc: {avg_acc:.4f} | Dice: {avg_dice:.4f} | IoU: {avg_iou:.4f}')

    return history

print('Training function ready.')

## 🚀 5. Run Training on Both Datasets

In [ ]:
sun_history = train_and_evaluate(sun_dataset, 'SUN Database')
ld_history  = train_and_evaluate(ld_dataset,  'LDPolypVideo')

## 📈 6. Results Comparison — Charts

In [ ]:
epochs_range = range(1, EPOCHS + 1)
metrics = [
    ('loss', 'Training Loss',        'Loss'),
    ('acc',  'Validation Accuracy',  'Accuracy'),
    ('dice', 'Dice Coefficient',     'Dice Score'),
    ('iou',  'IoU Score',            'IoU'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('SUN Database vs LDPolypVideo — U-Net Baseline Comparison', fontsize=15, fontweight='bold')

for ax, (key, title, ylabel) in zip(axes.flatten(), metrics):
    ax.plot(epochs_range, sun_history[key], 'o-', color='#2196F3', linewidth=2, markersize=6, label='SUN Database')
    ax.plot(epochs_range, ld_history[key],  's-', color='#FF5722', linewidth=2, markersize=6, label='LDPolypVideo')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_xticks(list(epochs_range))

plt.tight_layout()
plt.savefig('comparison_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as comparison_plot.png')

## 📋 7. Final Results Summary Table

In [ ]:
print('=' * 65)
print(f'{"Metric":<25} {"SUN Database":>18} {"LDPolypVideo":>18}')
print('=' * 65)

labels = [
    ('Best Train Loss',    'loss',  min),
    ('Best Val Accuracy',  'acc',   max),
    ('Best Dice Score',    'dice',  max),
    ('Best IoU Score',     'iou',   max),
    ('Final Train Loss',   'loss',  lambda x: x[-1]),
    ('Final Dice Score',   'dice',  lambda x: x[-1]),
    ('Final IoU Score',    'iou',   lambda x: x[-1]),
]

for label, key, fn in labels:
    sv = fn(sun_history[key])
    lv = fn(ld_history[key])
    print(f'{label:<25} {sv:>18.4f} {lv:>18.4f}')

print('=' * 65)
print(f'{"Total Samples":<25} {len(sun_dataset):>18,} {len(ld_dataset):>18,}')
print(f'{"Samples Used":<25} {min(SUBSET, len(sun_dataset)):>18,} {min(SUBSET, len(ld_dataset)):>18,}')
print('=' * 65)